In [5]:
from dotenv import load_dotenv

load_dotenv()

True

In [15]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader

llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0 #don't hallunicate, be deterministic
)

embeddings = GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-001'
)
pdf_path='my_resume.pdf'

if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF file '{pdf_path}' not found.")
pdf_loader = PyPDFLoader(pdf_path)

try:
    pages = pdf_loader.load()
    print(f'PDF loaded with {len(pages)} pages')
except Exception as e:
    print(f'Error loading PDF: {e}')
    raise



PDF loaded with 2 pages


In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.tools import tool

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=100
)

pages_split = text_splitter.split_documents(pages)

persist_directory = r'C:\Users\HP PROBOOK 430 G3\Documents\Internship_2024\langgraph\agents\db'
collection_name = 'resume'

if not os.path.exists(persist_directory):
    os.makedirs(persist_directory)
    
try:
    vectorstore = Chroma.from_documents(
        pages_split,
        embeddings,
        persist_directory=persist_directory,
        collection_name=collection_name
    )
except Exception as e:
    print(f'Error creating vectorstore: {e}')
    raise

# Now we create our retriever 
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5} # K is the amount of chunks to return
)

@tool
def retriever_tool(query: str) -> str:
    """This tool searches and returns the information from miracle nnadiukwu's resume.

    Args:
        query (str): search query

    Returns:
        str: _description_
    """
    
    docs = retriever.invoke(query)
    
    if not docs:
        return "I found no relevant information in miracle nnadiukwu's resume"
    
    results = []
    for i, doc in enumerate(docs):
        results.append(f"Document {i+1}:\n{doc.page_content}")
        
    return '\n\n'.join(results)

tools = [retriever_tool]

llm = llm.bind_tools(tools)


In [17]:
from typing import Annotated, TypedDict

from langgraph.graph import add_messages


class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    

def should_continue(state: AgentState) -> bool:
    """Check if last message contains a tool call

    Args:
        state (AgentState): graph's current state

    Returns:
        bool: flag indicating if cycle should continue 
    """
    result = state['messages'][-1]
    return hasattr(result, 'tool_calls') and len(result.tool_calls) > 0


system_prompt = """
You are an intelligent AI assistant who answers questions about Miracle Nnadiukwu and his qualifications based on the PDF document loaded into your knowledge base.
Use the retriever tool available to answer questions about Miracle Nnadiukwu. You can make multiple calls if needed.
If you need to look up some information before asking a follow up question, you are allowed to do that!
Please always cite the specific parts of the documents you use in your answers.
"""

In [18]:
from langchain_core.messages import SystemMessage, ToolMessage

tools_dict = {tool.name: tool for tool in tools}

def call_llm(state: AgentState) -> dict:
    """function to call LLM with the current state

    Args:
        state (AgentState): the graph's current state
    Returns:
        dict: changes made to the graph's state
    """
    
    messages = list(state['messages'])
    messages = [SystemMessage(content=system_prompt)] + messages
    message = llm.invoke(messages)
    return {'messages': [message]}

# Retriever Agent
def take_action(state: AgentState) -> dict:
    """Execute tool calls from the LLM's response."""

    tool_calls = state['messages'][-1].tool_calls
    results = []
    for t in tool_calls:
        print(f"Calling Tool: {t['name']} with query: {t['args'].get('query', 'No query provided')}")
        
        if not t['name'] in tools_dict: # Checks if a valid tool is present
            print(f"\nTool: {t['name']} does not exist.")
            result = "Incorrect Tool Name, Please Retry and Select tool from List of Available tools."
        
        else:
            result = tools_dict[t['name']].invoke(t['args'].get('query', ''))
            print(f"Result length: {len(str(result))}")
            

        # Appends the Tool Message
        results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))

    print("Tools Execution Complete. Back to the model!")
    return {'messages': results}

In [19]:
from langgraph.graph import START, END, StateGraph


graph = StateGraph(AgentState)
graph.add_node("llm", call_llm)
graph.add_node("retriever_agent", take_action)

graph.add_edge(START, 'llm')

graph.add_conditional_edges(
    "llm",
    should_continue,
    {True: "retriever_agent", False: END}
)
graph.add_edge("retriever_agent", "llm")

rag_agent = graph.compile()

In [22]:
from langchain_core.messages import HumanMessage

def running_agent():
    print("\n=== RAG AGENT===")
    
    while True:
        user_input = input("\nWhat is your question: ")
        if user_input.lower() in ['exit', 'quit']:
            break
            
        messages = [HumanMessage(content=user_input)] # converts back to a HumanMessage type

        result = rag_agent.invoke({"messages": messages})
        
        print("\n=== ANSWER ===")
        print(result['messages'][-1].content[0]['text'])

In [23]:
running_agent()


=== RAG AGENT===
Calling Tool: retriever_tool with query: Miracle Nnadiukwu's grade on graduation
Result length: 2372
Tools Execution Complete. Back to the model!

=== ANSWER ===
Miracle Nnadiukwu is expected to graduate with **First Class Honors** from Pan-Atlantic University with a Bachelor of Science in Computer Science. (Document 1)
Calling Tool: retriever_tool with query: Miracle Nnadiukwu masters primary focus
Result length: 2372
Tools Execution Complete. Back to the model!

=== ANSWER ===
Based on the provided document, there is no explicit information stating Miracle Nnadiukwu's intention to pursue a Master's degree.

His primary academic focus, as stated in his "Manuscript in Preparation | Occlusion-Sensitive Emotion Detection using GAN Augmentation (2025)", is the "evaluation of landmark-driven GAN pipelines for occlusion recovery in facial recognition and broader computer vision applications." (Document 2)

Professionally, he is focused on "Building a PropTech platform addr